# Reading CSV Files with Explicit Schema Definition

## Two Methods to Define Schema:

1. **Simple DDL String** - Quick SQL-like syntax
2. **StructType with StructField** - Programmatic with full control

### Why Define Schema Explicitly?
* ✓ **Performance** - No schema inference overhead
* ✓ **Data Quality** - Type enforcement at read time
* ✓ **Consistency** - Same schema across files
* ✓ **Control** - Nullable constraints and metadata

Let's explore both methods using real CSV data!

In [0]:
# Define CSV path
csv_path = "/Volumes/dev_finance/staging/data_export/csv/"

print(f"📁 CSV Location: {csv_path}")
print("\n⏳ Let's first read CSV with schema inference to understand the data...")

# Read without schema (Spark infers it)
df_inferred = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(csv_path)

print("\n📊 Inferred Schema:")
df_inferred.printSchema()

print(f"\n📈 Total Records: {df_inferred.count()}")
print("\n📄 Sample Data:")
display(df_inferred.limit(5))

## Method 1: Simple DDL String Schema

**DDL (Data Definition Language) String** is the quickest way to define a schema.

### Syntax:
```python
schema = "column1 TYPE, column2 TYPE, column3 TYPE"
```

### Advantages:
* Quick and easy to write
* Familiar SQL syntax
* Good for flat CSV structures

### Limitations:
* Cannot specify nullable constraints explicitly
* Cannot add metadata
* All fields default to nullable=true

In [0]:
# Method 1: Define schema using DDL String
ddl_schema = "review STRING, franchiseID INT, review_date TIMESTAMP, new_id INT"

print("✅ DDL Schema Defined:")
print(f"   {ddl_schema}")

# Read CSV with DDL schema
df_ddl = spark.read \
    .schema(ddl_schema) \
    .option("header", "true") \
    .csv(csv_path)

print("\n📊 DataFrame Schema (DDL Method):")
df_ddl.printSchema()

print("\n📄 Sample Data:")
display(df_ddl.limit(5))

print("\n✨ Notice: All fields are nullable=true by default")

## Method 2: StructType with StructField (Programmatic)

This method gives you **complete control** over every aspect of the schema.

### StructField Constructor:
```python
StructField(name, dataType, nullable=True, metadata=None)
```

### Four Key Properties:

| Property | Type | Description | Example |
|----------|------|-------------|----------|
| **name** | string | Column name | `"customer_id"`, `"email"` |
| **dataType** | DataType | Data type | `StringType()`, `IntegerType()` |
| **nullable** | boolean | Can be NULL? | `True` (default) or `False` |
| **metadata** | dict | Additional info | `{"description": "...", "pii": True}` |

### When to Use:
* Need NOT NULL constraints
* Want to add business metadata
* Building production pipelines
* Dynamic schema generation

In [0]:
# Method 2: Define schema using StructType and StructField
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

# Create explicit schema with full control
struct_schema = StructType([
    # Field 1: review - nullable, with metadata
    StructField(
        name="review",
        dataType=StringType(),
        nullable=True,
        metadata={"description": "Customer review text", "max_length": 1000}
    ),
    
    # Field 2: franchiseID - NOT nullable (required field)
    StructField(
        name="franchiseID",
        dataType=IntegerType(),
        nullable=False,  # ⚠️ This field is REQUIRED
        metadata={"description": "Unique franchise identifier", "business_key": True}
    ),
    
    # Field 3: review_date - nullable timestamp
    StructField(
        name="review_date",
        dataType=TimestampType(),
        nullable=True,
        metadata={"description": "When review was submitted", "timezone": "UTC"}
    ),
    
    # Field 4: new_id - nullable integer
    StructField(
        name="new_id",
        dataType=IntegerType(),
        nullable=True,
        metadata={"description": "Sequential review ID"}
    )
])

print("✅ StructType Schema Defined")
print("\n📋 Schema Structure:")
print(struct_schema.simpleString())

print("\n📝 Detailed Field Information:")
for field in struct_schema.fields:
    print(f"\n  🔹 {field.name}")
    print(f"     Type: {field.dataType.simpleString()}")
    print(f"     Nullable: {field.nullable}")
    print(f"     Metadata: {field.metadata}")

In [0]:
# Read CSV with StructType schema
df_struct = spark.read \
    .schema(struct_schema) \
    .option("header", "true") \
    .csv(csv_path)

print("📊 DataFrame Schema (StructType Method):")
df_struct.printSchema()

print("\n📄 Sample Data:")
display(df_struct.limit(5))

print("\n✨ Notice: franchiseID is nullable=false")
print("   Metadata is also preserved in the schema!")

In [0]:
# Side-by-side comparison
print("="*70)
print("COMPARISON: DDL String vs StructType")
print("="*70)

print("\n1️⃣ DDL String Schema:")
print(f"   Code: schema = \"{ddl_schema}\"")
print("   Pros: Quick, simple, SQL-like")
print("   Cons: All nullable=true, no metadata")

print("\n2️⃣ StructType Schema:")
print("   Code: StructType([StructField(...), ...])")
print("   Pros: Full control, NOT NULL constraints, metadata support")
print("   Cons: More verbose, requires imports")

print("\n" + "="*70)
print("RECORD COUNT COMPARISON")
print("="*70)
print(f"Inferred Schema: {df_inferred.count()} records")
print(f"DDL Schema:      {df_ddl.count()} records")
print(f"StructType:      {df_struct.count()} records")

print("\n" + "="*70)
print("RECOMMENDATION")
print("="*70)
print("✓ Use DDL String: Quick prototyping, simple CSV files")
print("✓ Use StructType: Production pipelines, data quality enforcement")

In [0]:
# Write DataFrame with explicit schema to Delta table
target_table = "dev_finance.staging.bakehouse_reviews"

print(f"💾 Writing to Delta Table: {target_table}")
print("   Using StructType schema for data quality enforcement\n")

# Write with explicit schema
df_struct.write \
    .mode("overwrite") \
    .option("mergeSchema", "false") \
    .saveAsTable(target_table)

print("✅ Data written successfully!")

# Verify
print(f"\n🔍 Verifying table: {target_table}")
df_verify = spark.table(target_table)
df_verify.printSchema()

print(f"\n📊 Row Count: {df_verify.count()}")
print("\n📄 Sample Records:")
display(df_verify.limit(5))

## 📚 Summary: CSV Schema Definition Methods

### Method 1: DDL String
```python
schema = "col1 STRING, col2 INT, col3 TIMESTAMP"
df = spark.read.schema(schema).option("header", "true").csv(path)
```
**Use when:** Quick prototyping, simple CSV files

### Method 2: StructType & StructField
```python
schema = StructType([
    StructField("col1", StringType(), nullable=True, metadata={...}),
    StructField("col2", IntegerType(), nullable=False)
])
df = spark.read.schema(schema).option("header", "true").csv(path)
```
**Use when:** Production pipelines, NOT NULL constraints needed

---

## 🎯 Key Takeaways

1. **DDL String** = Fast and simple (all nullable=true)
2. **StructType** = Full control (nullable constraints + metadata)
3. **Explicit schemas** = Better performance than inference
4. **Metadata** = Document business rules and data quality requirements

---

## 🔜 Next Topics

Now that you understand CSV schema definition, we can extend this to:

* **JSON files** - Handle nested structures with StructType
* **Parquet files** - Binary format with embedded schema
* **Complex types** - Arrays, Maps, Nested Structs

Ready to move forward? 🚀

In [0]:
# Define the path to CSV files
path = "/Volumes/dev_finance/staging/data_export/csv/"

print(f"📁 CSV Path: {path}")
print("✓ Path variable ready for use in schema examples")

   
## Two Ways to Define Schemas in PySpark

### Method 1: Simple Schema (DDL String)
- Quick and concise
- Uses SQL-like syntax
- Good for simple, flat structures

### Method 2: StructType with StructField (Programmatic)
- More control and flexibility
- Required for complex nested structures
- Better for dynamic schema generation
- Allows adding metadata

Let's explore both methods with the same CSV data!

In [0]:
# Method 1: Simple Schema Definition using DDL String
# This is the quickest way to define a schema

simple_ddl_schema = "review STRING, franchiseID INT, review_date TIMESTAMP, new_id INT"

print("✓ Simple DDL Schema (String-based):")
print(f"  {simple_ddl_schema}")
print("\n📝 Advantages:")
print("  - Quick and easy to write")
print("  - Familiar SQL syntax")
print("  - Good for flat structures")
print("\n⚠️ Limitations:")
print("  - Cannot specify nullable constraints explicitly")
print("  - Cannot add metadata")
print("  - Less suitable for complex nested structures")

# Read CSV with simple schema
df_simple = spark.read.schema(simple_ddl_schema).option("header", "true").csv(path)

print("\n📊 DataFrame Schema:")
df_simple.printSchema()

print("\n📄 Sample Data:")
display(df_simple.limit(3))

   
## Method 2: StructType with StructField (Programmatic Approach)

### StructField Constructor:
```python
StructField(name, dataType, nullable=True, metadata=None)
```

### Four Key Properties:

1. **name** (string) - Required
   - The column name
   - Example: `"customer_id"`, `"email"`, `"amount"`

2. **dataType** (DataType) - Required
   - The data type for this field
   - Examples: `StringType()`, `IntegerType()`, `DoubleType()`
   - Can be complex: `ArrayType()`, `MapType()`, or nested `StructType()`

3. **nullable** (boolean) - Optional (default: True)
   - Whether this field can contain NULL values
   - `True`: NULL values allowed
   - `False`: NULL values NOT allowed (will throw error)

4. **metadata** (dict) - Optional (default: None)
   - Additional information about the field
   - Common uses: descriptions, data quality rules, business logic
   - Example: `{"description": "Customer email address", "pii": True}`

In [0]:
# Method 2: Explicit Schema using StructType and StructField
# This gives you complete control over every property

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

# Define schema with all four properties for each field
explicit_schema = StructType([
    # Property 1: name = "review"
    # Property 2: dataType = StringType()
    # Property 3: nullable = True (can be NULL)
    # Property 4: metadata = {description, business context}
    StructField(
        "review", 
        StringType(), 
        nullable=True,
        metadata={"description": "Customer review text", "max_length": 1000}
    ),
    
    # Franchise ID - NOT nullable (required field)
    StructField(
        "franchiseID", 
        IntegerType(), 
        nullable=False,  # This field is REQUIRED
        metadata={"description": "Unique franchise identifier", "business_key": True}
    ),
    
    # Review timestamp
    StructField(
        "review_date", 
        TimestampType(), 
        nullable=True,
        metadata={"description": "When review was submitted", "timezone": "UTC"}
    ),
    
    # New ID field
    StructField(
        "new_id", 
        IntegerType(), 
        nullable=True,
        metadata={"description": "Sequential review ID"}
    )
])

print("✓ Explicit Schema with StructType & StructField:")
print("\n📋 Schema Structure:")
print(explicit_schema.simpleString())

print("\n📝 Schema Details:")
for field in explicit_schema.fields:
    print(f"\n  Field: {field.name}")
    print(f"    ├─ DataType: {field.dataType.simpleString()}")
    print(f"    ├─ Nullable: {field.nullable}")
    print(f"    └─ Metadata: {field.metadata}")

In [0]:
# Read the same CSV using our explicit StructType schema
df_explicit = spark.read.schema(explicit_schema).option("header", "true").csv(path)

print("📊 DataFrame with Explicit Schema:")
df_explicit.printSchema()

print("\n📄 Sample Data:")
display(df_explicit.limit(3))

print("\n🔍 Key Differences:")
print("  ✓ Method 1 (DDL String): Fast, simple, less control")
print("  ✓ Method 2 (StructType): More verbose, full control, metadata support")

print("\n💡 When to use StructType:")
print("  • Need to enforce NOT NULL constraints")
print("  • Want to add business metadata")
print("  • Building complex nested structures")
print("  • Generating schemas dynamically")
print("  • Need to validate data quality requirements")

In [0]:
# Advanced: Creating a nested structure
# Let's create a more complex schema with nested objects

from pyspark.sql.types import ArrayType, MapType

advanced_schema = StructType([
    StructField("review_id", IntegerType(), False),
    
    # Nested struct for franchise information
    StructField("franchise_info", StructType([
        StructField("franchise_id", IntegerType(), False),
        StructField("location", StringType(), True),
        StructField("city", StringType(), True)
    ]), nullable=True, metadata={"description": "Franchise location details"}),
    
    # Array of tags
    StructField("tags", ArrayType(StringType()), True,
                metadata={"description": "Review categories/tags"}),
    
    # Map for ratings by category
    StructField("ratings", MapType(StringType(), IntegerType()), True,
                metadata={"description": "Category-wise ratings"}),
    
    # Main review text
    StructField("review_text", StringType(), True),
    
    StructField("review_date", TimestampType(), True)
])

print("🏗️ Advanced Schema with Nested Structures:")
print(advanced_schema.simpleString())

print("\n📚 This schema demonstrates:")
print("  • StructType inside StructType (nested object)")
print("  • ArrayType for collections")
print("  • MapType for key-value pairs")
print("  • Complex metadata")

# Create sample data matching this schema
from datetime import datetime

sample_nested_data = [
    (1, (3000017, "Tenmonkan", "Kagoshima"), ["cookies", "bakery"], {"taste": 5, "service": 4}, "Great cookies!", datetime(2024, 5, 20)),
    (2, (3000018, "East 6th Street", "Austin"), ["dessert", "sweet"], {"taste": 4, "service": 5}, "Sweet heaven!", datetime(2024, 5, 21))
]

df_nested = spark.createDataFrame(sample_nested_data, advanced_schema)

print("\n📊 Nested DataFrame:")
df_nested.printSchema()

print("\n📄 Sample Nested Data:")
display(df_nested)

In [0]:
# Write the DataFrame to Delta table in Unity Catalog
# Using the explicit schema ensures data quality

target_table = "dev_finance.staging.bakehouse_reviews"

print(f"💾 Writing data to Delta table: {target_table}")

# Write with explicit schema enforcement
df_explicit.write \
    .mode("overwrite") \
    .option("mergeSchema", "false") \
    .saveAsTable(target_table)

print(f"\n✅ Data written successfully!")
print(f"\n🔍 Verify the table:")

# Read back and verify
df_verify = spark.table(target_table)
df_verify.printSchema()

print(f"\n📊 Row count: {df_verify.count()}")
print("\n📄 Sample records:")
display(df_verify.limit(5))

print("\n✨ Benefits of Explicit Schema:")
print("  ✓ Data type enforcement at load time")
print("  ✓ NULL constraint validation")
print("  ✓ Metadata preserved for documentation")
print("  ✓ Consistent structure across loads")
print("  ✓ Early error detection")

   
## 📚 Summary: When to Use Each Method

### Use DDL String (Method 1) when:
* ✓ Schema is simple and flat
* ✓ Quick prototyping or exploration
* ✓ You don't need nullable constraints
* ✓ No metadata required

### Use StructType & StructField (Method 2) when:
* ✓ Need to enforce NOT NULL constraints
* ✓ Adding business metadata (descriptions, rules)
* ✓ Complex nested structures (arrays, maps, nested structs)
* ✓ Generating schemas dynamically from external sources
* ✓ Data quality validation is critical
* ✓ Building production data pipelines

## 🎯 Key Takeaways

1. **Explicit schemas prevent silent data errors** - Type mismatches fail early
2. **StructField gives you 4 properties** - name, dataType, nullable, metadata
3. **Metadata is your friend** - Document schemas for future maintainers
4. **NOT NULL constraints save debugging time** - Fail fast on bad data
5. **Nested structures require StructType** - DDL strings are limited

## 🔗 Common Data Types Reference

| DataType | Use Case | Example |
|----------|----------|----------|
| `StringType()` | Text data | Names, descriptions, IDs |
| `IntegerType()` | 32-bit integers | Counts, IDs, small numbers |
| `LongType()` | 64-bit integers | Large counts, timestamps |
| `DoubleType()` | Decimal numbers | Prices, measurements |
| `BooleanType()` | True/False | Flags, status indicators |
| `DateType()` | Date only | Birth dates, event dates |
| `TimestampType()` | Date + Time | Transaction times, logs |
| `ArrayType(T)` | Collections | Lists of items |
| `MapType(K,V)` | Key-value pairs | Configurations, attributes |
| `StructType([...])` | Nested objects | Addresses, complex entities |